In [ ]:

#naming is off becuse of copy pasting to save timeimport numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')



In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
Delivery_path = os.path.join(path, 'Q3_data.csv')
df_cars = pd.read_csv(Delivery_path)

print(f"Dataset shape: {df_cars.shape}")

In [ ]:
# Task 2: Write your code here:
df_cars.head()

In [ ]:
# Task 3: Write your code here:
df_cars.info()

In [ ]:
# Task 4: Write your code here:
df_cars.describe()

In [ ]:
# Task 1: Write your code here:
stat_cols = ['P_2', 'D_39', 'B_1', 'B_2', 'R_1']
df_clean = df_cars.dropna().copy()
for col in stat_cols:
    df_clean[col] = df_clean[col].fillna('unknown')
print("Missing values remaining:", df_clean.isnull().sum().sum())

In [ ]:
# Task 2: Write your code here:
def check_duplicates(df_clean):

  #TODO: get duplicated data using pandas
  duplicates = df_clean.duplicated().sum()

  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df_clean.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 3: Write your code here:
categorical_cols = ['P_2', 'D_39', 'B_1', 'B_2', 'R_1']
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

df_clean.head()

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import MinMaxScaler

# Pick only the numerical columns, NOT the target
numerical_cols = df_clean.select_dtypes(include=["number"]).columns.drop()

scaler = MinMaxScaler()

# scale the `numerical_cols`
df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])

df_clean.head()


In [ ]:
# Task 5: Write your code here:
def check_target_imbalance(df_clean):
  print("Target Distribution:")

  df_clean[categorical_cols].hist()  # Yeah you can just do this :)
  plt.show()

check_target_imbalance(df_clean)

In [ ]:
feature_cols = ['P_2', 'D_39', 'B_1', 'B_2', 'R_1']
X = df_clean[feature_cols]
y = df_clean[ 'P_2']

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

sr_results = {'loss': [], 'acc': [], 'f1': []}

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):

  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # Get the train & test split for this fold
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train using gradient descent with learning rate = 0.5
  theta, losses = gradient_descent(X_train, y_train, lr=0.5, num_classes=4)

  # Calculate z & class probabilities for X_test
  z = np.dot(X_test, theta)
  y_pred_proba = softmax(z)

  # Pick the predicted classes with the highest probability
  y_pred = np.argmax(y_pred_proba, axis=1)

  # Calculate evaluation metrics
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, average='macro')  # for multiclass f1 score, you should set the average hyperparameter ("macro", "micro", "weighted")

  # Store results
  sr_results['loss'].append(losses)
  sr_results['acc'].append(accuracy)
  sr_results['f1'].append(f1)

In [ ]:
pip install catboost

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [ ]:
# TODO: Define models with hyperparameters of your choice
models = {
  "LR": LogisticRegression(
      max_iter=1000
  ),
  "KNN": KNeighborsClassifier(
      n_neighbors=3,
  ),
  "SVM": SVC(
      kernel='rbf',
      C=1.0
  ),
  "Decision Tree": DecisionTreeClassifier(
      max_depth=16
  ),
  "Random Forest": RandomForestClassifier(
      n_estimators=200,
      max_depth=10
  ),
  "XGBoost": XGBClassifier(
      verbosity=0,
      n_estimators=280,
      max_depth=10,
  ),
  "CatBoost": CatBoostClassifier(
      verbose=0,
      n_estimators=200,
      max_depth=4
  )
}

In [ ]:
print("Evaluating models on the test set...")

# Dictionary to store performance metrics
model_performance = {}

# Iterate through each trained model
for model_name, model in models.items():
    print(f"\nEvaluating {model_name}...")

    # Make predictions on the test set (hard labels)
    y_pred = model.predict(X_test)

    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    # Store metrics
    model_performance[model_name] = {
        'Accuracy': accuracy,
        'F1-Score': f1
    }

    # Print metrics
    print(f"  Accuracy: {accuracy:.4f}")

    print(f"  F1-Score: {f1:.4f}")

    # Generate and visualize Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
    plt.title(f'Confusion Matrix for {model_name}')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.show()

print("\nAll models evaluated.")

In [ ]:
# Task 1: Write your code here:

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: